# XRD Rietveld Plot Generator

Publication-quality Rietveld plots from the **CSV that the GSAS-II Rietveld
plot saves** - batch processing, built-in validation, cross-platform.

**Fix the header of each export once, before you run anything.** GSAS-II
writes one header name more than it writes data fields, so every name sits
one column left of its own data: the angles arrive under `used` and the
counts arrive under `x, 2theta (deg)`. Delete the `used` cell of the header
row in a spreadsheet and shift that row one place left. Section 3 refuses a
file still shifted, since the figure would look convincing and be wrong.

**Then leave the header names alone, except the phase columns.** The pattern
columns are found by their names: `obs` renamed is drawn flat, and a renamed
`diff/sigma` stops the file. The phase columns are yours to rename, and that
is the only place your phase names come from, so `Phase 1` becomes `Rutile`
in the legend by editing that header alone.

Not the file from *Export → Powder data as → histogram CSV file*: that one
has a quoted preamble and different column names, and is rejected.

Full documentation (input format, usage, configuration, privacy notes):
see [`README.md`](README.md) and
[`docs/input-format.md`](docs/input-format.md).

## 1. Setup

Dependency check, then the engine. Parsing, data preparation, plotting and
the batch driver live in [`xrd_plotter.py`](xrd_plotter.py), imported here
as `xp`. Plot appearance (2θ window, colours, line widths, fonts) is set by
the constants at the top of that file and overridden on the module, as the
cell below shows. Input format and numerical-precision details are in the
README.

In [ ]:
# Dependency bootstrap - installs only what is missing. IPython arrives with
# any Jupyter kernel, and is listed so an editor resolves it as well.
import importlib.util, subprocess, sys

for module, package in (("numpy", "numpy"), ("pandas", "pandas"),
                        ("matplotlib", "matplotlib"),
                        ("ipywidgets", "ipywidgets"), ("IPython", "ipython"),
                        ("pytest", "pytest")):
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "--quiet", package])
print("Dependencies OK")

In [ ]:
"""The plotting engine lives in xrd_plotter.py; this cell loads it."""
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xrd_plotter as xp

# Appearance is set by the constants in the module. Override them here, on
# the module itself, so every routine sees the change:
#   xp.PLOT_X_MIN, xp.PLOT_X_MAX = 13, 85   # fix the 2theta window
#   xp.WEIGHTED_RESIDUALS = False           # raw diff in the lower panel
#   xp.PHASE_COLORS = {"phase 1": "#1f77b4"}
#   xp.PREVIEW_WIDTH_PX = 500               # smaller inline previews in sec. 3
print("Engine loaded:", Path(xp.__file__).name)


## 2. Validation (self-test on synthetic data)

Runs [`test_xrd_plotter.py`](test_xrd_plotter.py) on synthetic
GSAS-II-style exports. It checks bit-exact parsing across separator and
decimal-mark variants, phase-column detection in a full export, refusal of
an export whose header is one place out of step, isolation of corrupt,
ragged and incomplete files, phase order and colours, the metadata legend,
the 2θ window, the axis labels and ticks, the unweighted residual panel, the
interactive helper, and a batch surviving one unusable file.

The cell stops at the first failing assertion, so running the notebook is a
test run, as is `pytest -q` from a terminal. Only synthetic data is used.

In [ ]:
import pytest

# The suite builds its own synthetic exports, so this cell reads nothing
# from data/ and works on a fresh clone.
exit_code = pytest.main(["-q", "--no-header", "test_xrd_plotter.py"])
assert exit_code == 0, "the validation suite failed, see the report above"
print("\nALL VALIDATION CHECKS PASSED")

## 3. Plot your own exports

Copy your CSV exports into `data/`, optionally place `Samples_metadata.csv`
next to the notebook, set the four values below and run. Each export needs
the header fix described at the top of this notebook first.

Each file gets one block: its name, the phases found with the colour each
one was drawn in, the 2θ window used and where it came from (`metadata` or
`auto`, the full measured range), the figure, then the two files written to
`output/`. The PDF goes to `output/pdf/` and the PNG to `output/png/`, sorted
by extension. A file the engine fails to draw prints `FAILED` with the reason
and the run continues. Every failure is repeated in the summary at the end.

A file whose header is still one place out of step prints `FAILED: the
'x, 2theta (deg)' column rises and falls`. Fix that file and run again.

The `x_min` / `x_max` columns set the window here, per sample. Section 4
opens each file on the same window, prefilled from these columns; clear a
box there to widen back to the full range.

> **Keep your data private:** `data/`, `output/` and `Samples_metadata.csv`
> are listed in `.gitignore` and must never be committed or uploaded.

In [ ]:
DATA_FOLDER = "data"                       # your GSAS-II CSV exports
METADATA_FILE = "Samples_metadata.csv"     # optional, PRIVATE - never commit
OUTPUT_FOLDER = "output"                   # created automatically
USE_SQRT = True                            # False -> linear intensity axis

results = xp.process_folder(DATA_FOLDER, METADATA_FILE, OUTPUT_FOLDER,
                         use_sqrt=USE_SQRT)

## 4. Try a different window on one file

Pick a file and type limits. The figure updates live and in place: a text
box on Enter or when you leave it, a checkbox or the dropdown at once.
Picking a file fills the 2θ boxes from its metadata row, so it opens on the
same window section 3 draws; clear a box for the full measured range.

**Save to output** writes the window on screen to `output/` at full
resolution, under the same name the batch uses, so you can settle a window
here without rerunning section 3. Every other control only previews.

An empty box leaves its end of the axis to the setting behind it: the
`xp.PLOT_X_MIN` and `xp.PLOT_X_MAX` constants for 2θ, the data itself for
intensity.

The line above the figure carries the engine messages for this file and the
metadata row for the window on screen. Paste the row into
`Samples_metadata.csv` and section 3 draws the sample this way every run.

Needs `ipywidgets`, installed by the first cell. Without it the section
prints how to install it, and the rest of the notebook is unaffected.

In [ ]:
# No widgets.Output here. An Output cleared inside a callback appends a second
# figure under the first in VS Code, so the panel would fill with stale plots.
# The figure is a widgets.Image and the log a widgets.HTML, both value
# replaced, so every change updates the same two areas in place, never appends.
import contextlib
import html
import io
import traceback

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    print("ipywidgets is not installed: run 'pip install ipywidgets', "
          "then re-run this cell.")

files = sorted(f for f in Path(DATA_FOLDER).glob("*.csv")
               if f.name != Path(METADATA_FILE).name)


def pre(text):
    """Escaped monospace block for a widgets.HTML value."""
    return ("<pre style='margin:0;font:12px/1.4 monospace;white-space:pre-wrap'>"
            f"{html.escape(text)}</pre>")


if widgets is None or not files:
    if widgets is not None:
        print(f"No CSV files in '{DATA_FOLDER}': nothing to replot.")
else:
    picker = widgets.Dropdown(options=[(f.name, str(f)) for f in files],
                              description="File:",
                              layout=widgets.Layout(width="420px"))
    # continuous_update=False: a box redraws when you press Enter or leave
    # it, not on every keystroke.
    boxes = {k: widgets.Text(description=d, placeholder="auto",
                             continuous_update=False,
                             layout=widgets.Layout(width="180px"))
             for k, d in (("x_min", "2theta min"), ("x_max", "2theta max"),
                          ("y_min", "y min"), ("y_max", "y max"))}
    sqrt_box = widgets.Checkbox(value=USE_SQRT, description="sqrt intensity")
    weighted_box = widgets.Checkbox(value=xp.WEIGHTED_RESIDUALS,
                                    description="diff/sigma")
    save_button = widgets.Button(description="Save to output",
                                 button_style="success", icon="download")
    status = widgets.HTML()
    canvas = widgets.Image(format="png",
                           layout=widgets.Layout(width="100%",
                                                 max_width="820px"))
    busy = [False]
    filling = [False]  # True while a file pick rewrites the 2theta boxes

    def draw_current():
        """Render the picked file with whatever the controls now hold.

        Returns (fig, metadata_line, log_text). Raises ValueError when the
        file cannot be drawn. The caller decides whether to preview or save.
        """
        limits = {k: xp.to_number(b.value) if b.value.strip() else None
                  for k, b in boxes.items()}
        log = io.StringIO()
        # Engine messages belong in the status block. Left on stdout they
        # land under the cell and pile up one copy per redraw.
        with contextlib.redirect_stdout(log):
            fig, line = xp.replot_file(picker.value, METADATA_FILE,
                                       use_sqrt=sqrt_box.value,
                                       weighted=weighted_box.value, **limits)
        return fig, line, log.getvalue()

    def redraw(_=None):
        """Update the on-screen preview in place; saves nothing."""
        if filling[0] or busy[0]:
            return  # skip the box writes of a file pick, and re-entrancy
        busy[0] = True
        try:
            try:
                fig, line, log = draw_current()
            except ValueError as e:
                # Hide the stale figure: it belongs to a different file, and
                # leaving it up invites new limits typed against it.
                canvas.layout.display = "none"
                status.value = pre(f"Cannot draw this file: {e}")
                return
            png = io.BytesIO()
            try:
                # 110 dpi is a screen preview. Save writes the 600 dpi file.
                fig.savefig(png, format="png", dpi=110, bbox_inches="tight",
                            facecolor="white")
            finally:
                plt.close(fig)  # a failed render must not leak the figure
            canvas.value = png.getvalue()
            canvas.layout.display = ""
            status.value = pre(f"{log}Metadata line for this window:\n{line}")
        except Exception:
            plt.close("all")
            canvas.layout.display = "none"
            status.value = pre("Redraw failed:\n" + traceback.format_exc())
        finally:
            busy[0] = False

    def on_pick(_=None):
        """Prefill the 2theta boxes from the file's metadata, then redraw.

        The boxes open on the window the batch would use, so a peak cropped
        out in section 3 is cropped here too. Clear a box to widen back.
        """
        filling[0] = True
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                x_min, x_max = xp.sample_window(METADATA_FILE,
                                                Path(picker.value).name)
            boxes["x_min"].value = "" if x_min is None else f"{x_min:g}"
            boxes["x_max"].value = "" if x_max is None else f"{x_max:g}"
        finally:
            filling[0] = False
        redraw()

    def save(_=None):
        """Write the current window to output/pdf and output/png."""
        if busy[0]:
            return
        busy[0] = True
        try:
            try:
                fig, _line, log = draw_current()
            except ValueError as e:
                status.value = pre(f"Cannot draw this file: {e}")
                return
            try:
                base = xp.output_basename(Path(picker.value).stem,
                                          sqrt_box.value, weighted_box.value)
                xp.save_figure(fig, OUTPUT_FOLDER, base)
            finally:
                plt.close(fig)
            status.value = pre(f"{log}Saved pdf/{base}.pdf and png/{base}.png")
        except Exception:
            plt.close("all")
            status.value = pre("Save failed:\n" + traceback.format_exc())
        finally:
            busy[0] = False

    # Picking a file refills the 2theta boxes and redraws; every other
    # control redraws the one figure in place. Only the button writes to disk.
    picker.observe(on_pick, names="value")
    for control in (sqrt_box, weighted_box, *boxes.values()):
        control.observe(redraw, names="value")
    save_button.on_click(save)

    display(widgets.VBox([
        picker,
        widgets.HBox([boxes["x_min"], boxes["x_max"]]),
        widgets.HBox([boxes["y_min"], boxes["y_max"]]),
        widgets.HBox([sqrt_box, weighted_box, save_button]),
        status,
        canvas,
    ]))
    on_pick()  # open on the first file, boxes prefilled from its metadata
